# T5 - Receive Beamforming (standalone)
Input: `received_pressure[element, time]` + `t_axis` + element positions. Output: beamformed RF (depth x lines). Here the input is synthetic (three point targets); to use real hand-off data set `HANDOFF = 'path/to/receiver_bf_input.npz'`.

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
import receiver_beamforming as rb
from receiver_beamforming import RxConfig

In [ ]:
HANDOFF = None    # e.g. ROOT/'integration/03_full_pipeline/outputs/.../receiver_bf_input.npz'
c, fs, f0, n = 1540.0, 40e6, 5e6, 128
ex = (np.arange(n) - (n - 1) / 2) * c / f0 / 2
t = np.arange(-1.2e-6, 50e-6, 1 / fs)
rx = np.zeros((n, t.size))
for x, z in [(-3e-3, 12e-3), (0.0, 22e-3), (3e-3, 32e-3)]:
    tau = z / c + np.hypot(x - ex, z) / c
    d = t[None, :] - tau[:, None]
    rx += np.sin(2*np.pi*f0*d) * np.exp(-0.5 * (d / (3/(2.5*f0)))**2)
print("synthetic channel data:", rx.shape)

In [ ]:
inp = rb.load_receiver_input(HANDOFF) if HANDOFF else rb.make_receiver_input(rx, t, ex, fs_hz=fs, c0=c, f0_hz=f0)
res = rb.beamform(inp, RxConfig(apodization='hann', f_number=1.5))
print('beamformed_rf', res.beamformed_rf.shape, '| extent mm', res.extent_mm)
print(res.notes)

In [ ]:
plt.figure(figsize=(5, 7)); plt.imshow(res.bmode_db, extent=res.extent_mm, cmap='gray', vmin=-50, vmax=0, aspect='auto')
plt.xlabel('x [mm]'); plt.ylabel('z [mm]'); plt.title('beamformed preview'); plt.colorbar(label='dB'); plt.show()

## Compare apodisation windows on the same data

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 5))
for a, apod in zip(ax, ['rect', 'hann', 'blackman']):
    b = rb.beamform(inp, RxConfig(apodization=apod))
    a.imshow(b.bmode_db, extent=b.extent_mm, cmap='gray', vmin=-50, vmax=0, aspect='auto'); a.set_title(apod)
plt.tight_layout(); plt.show()

## Are the targets where we put them?

In [ ]:
for x, z in [(-3e-3, 12e-3), (0.0, 22e-3), (3e-3, 32e-3)]:
    xb, zb = rb.locate_peak(res, x, z)
    print(f'true ({x*1e3:5.1f},{z*1e3:5.1f}) mm  error ({(xb-x)*1e3:+.3f},{(zb-z)*1e3:+.3f}) mm')